# Multi-Task v4 Kendall — 5-Fold CV (Q-01, thesis-number gate)

**Resolves [[Open_Questions]] Q-01.** Reports rigorous mean±SD on the v4 Kendall recipe's dice and cobb_direct MAE across 5 folds. After Q-18 closure (5 independent attempts confirmed v4 Kendall has reached the architecture's cobb floor at 5.79° on a single 80/20 split), this experiment locks that number with a proper confidence interval.

**What changes vs `multitask_v4.ipynb` (single 80/20)**:
- Replace `split_train_val(...)` with `KFold(n_splits=5, shuffle=True, random_state=SEED)` over `TRAINABLE`
- Loop 5 times: fresh model per fold, same v4 recipe, per-fold cache
- Aggregate metrics: mean ± SD across the 5 folds → **the headline thesis numbers**
- Optional cells (post-training, no retraining required):
  - **TTA**: hflip-only ensemble at inference (free 2× ensemble; hflip preserves `|Cobb|`)
  - **K-fold ensemble**: pool all 5 val sets; for each sample, average predictions from all 5 fold-models
  - **EMA**: maintained per-fold during training; optional alternative checkpoint to evaluate

**What stays identical to v4**:
- Architecture (`MultiTaskEncoderUNet`, ResNet-34, in_ch=1)
- Loss (`MixedLoss` = clamped Kendall on seg+cobb + fixed `λ_kpt=1.0`)
- Augmentation (`augment_v5_cobb_aware`)
- Schedule (150 ep, patience 25, encoder freeze warmup 10 ep, dual-checkpoint dice/cobb)
- Bare `/255` preprocessing (Tier-A rejected per [[2026-05-03_v4_tierA]])

**Compute estimate**: 5 folds × ~50 min/fold = **~4–5 hours total** on DirectML. Per-fold caching means a failed mid-run can resume by skipping completed folds.

**Acceptance gate** (the headline thesis number is whatever falls out of CV — no pass/fail bar):
- If mean cobb_direct MAE ≈ 5.79° ± 1° → single-split was representative; 5.79° is the architecture ceiling
- If mean ≈ 5.79° but high SD (>1.5°) → some folds got lucky/unlucky; ceiling is honest at the higher SD bound
- If mean significantly worse than 5.79° (>7°) → single-split overestimated the architecture; revise expectations

Cache key: `ai/models/checkpoints/multitask_v4_5fold/<timestamp>_<hash>/fold_K/` for K=0..4.


## 0 · Setup

In [2]:
import math
import sys
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import yaml
from sklearn.model_selection import KFold
from torch.utils.data import DataLoader, Dataset
from torchvision import models

warnings.filterwarnings("ignore", message=".*lerp.*")


In [3]:
def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "params.yaml").exists():
            return p
    raise FileNotFoundError("params.yaml not found above CWD")

REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

PARAMS = yaml.safe_load((REPO_ROOT / "params.yaml").read_text())
SEED = int(PARAMS["data"]["random_seed"])

torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    try:
        import torch_directml
        DEVICE = torch_directml.device()
    except ImportError:
        DEVICE = torch.device("cpu")
print(f"repo={REPO_ROOT}\ndevice={DEVICE}\nseed={SEED}")

repo=/home/ortiz/scoliosis
device=privateuseone:0
seed=42


In [4]:
from ai.evaluation.cobb import cobb_from_segmentation_tangent
from ai.preprocessing.keypoints import (
    KEYPOINTS_PER_VERTEBRA,
    TOTAL_KEYPOINTS,
    multiclass_mask_to_keypoints,
)
from ai.preprocessing.segmentation import NUM_SEG_CLASSES
from ai.training.augmentation import (
    affine_transform,
    clahe_like,
    coarse_dropout,
    elastic_deform,
    gamma_correction,
    gaussian_blur,
    gaussian_noise,
    horizontal_flip,
    intensity_jitter,
)
from ai.training.dataset import (
    IMG_H,
    IMG_W,
    TARGET_IDS_V2,
    preprocess_case,
)
from ai.training.losses import seg_loss_fn

NUM_VERT = len(TARGET_IDS_V2)
print(f"size={IMG_H}x{IMG_W}  seg_classes={NUM_SEG_CLASSES}  vertebrae={NUM_VERT}  keypoints={TOTAL_KEYPOINTS}")


size=512x256  seg_classes=18  vertebrae=17  keypoints=68


In [5]:
CLEAN_INDEX_CSV = REPO_ROOT / "data" / "processed" / "audit_v2_corrected" / "clean_index.csv"

def load_clean_index(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"{path} missing — run audit generation first")
    return pd.read_csv(path)

def trainable_rows(df: pd.DataFrame, min_vertebrae: int = 14) -> pd.DataFrame:
    mask = df["status"].isin(["ok", "warn"])
    if "target_vertebrae_count" in df.columns:
        mask = mask & (df["target_vertebrae_count"] >= min_vertebrae)
    return df[mask].reset_index(drop=True)

CLEAN_INDEX = load_clean_index(CLEAN_INDEX_CSV)
TRAINABLE = trainable_rows(CLEAN_INDEX)
n_with_cobb = TRAINABLE["cobb_angle_deg"].notna().sum()
print(f"total={len(CLEAN_INDEX)}  trainable={len(TRAINABLE)} (cobb_gt={n_with_cobb})")


total=250  trainable=249 (cobb_gt=178)


## 1 · 5-Fold Split Definition

`sklearn.model_selection.KFold(n_splits=5, shuffle=True, random_state=SEED)` over the TRAINABLE rows. Same SEED as the v4 single-split run, so fold 0's val is not necessarily v4's val (KFold uses different indexing than `df.sample(frac=1.0).iloc[:n_val]`) — but the data partition is reproducible.

Stratification on scoliosis/normal label is NOT applied here — the trainable set has 82 scoliosis + 70 normal, roughly balanced; random KFold should give similar ratios per fold (~16:14 each). If a future run shows imbalanced folds, switch to `StratifiedKFold` keyed on `grupo` (Scoliosis vs Normal).


In [6]:
N_FOLDS = 5
kfold = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLD_SPLITS: list[tuple[np.ndarray, np.ndarray]] = list(kfold.split(np.arange(len(TRAINABLE))))

print(f"K-fold splits ({N_FOLDS} folds, n_total={len(TRAINABLE)}):")
for k, (tr, va) in enumerate(FOLD_SPLITS):
    fold_train = TRAINABLE.iloc[tr]
    fold_val = TRAINABLE.iloc[va]
    n_scoliosis_val = (fold_val["category"] == "Scoliosis").sum() if "category" in fold_val else "?"
    n_with_cobb_val = fold_val["cobb_angle_deg"].notna().sum()
    print(f"  fold {k}: train={len(tr)}  val={len(va)} (scoliosis_val={n_scoliosis_val}, cobb_gt_val={n_with_cobb_val})")


K-fold splits (5 folds, n_total=249):
  fold 0: train=199  val=50 (scoliosis_val=32, cobb_gt_val=32)
  fold 1: train=199  val=50 (scoliosis_val=39, cobb_gt_val=39)
  fold 2: train=199  val=50 (scoliosis_val=35, cobb_gt_val=35)
  fold 3: train=199  val=50 (scoliosis_val=35, cobb_gt_val=35)
  fold 4: train=200  val=49 (scoliosis_val=37, cobb_gt_val=37)


## 1 · I/O Contract — Three Targets per Sample

| Tensor | Shape | Dtype | Source | Notes |
|---|---|---|---|---|
| `image` | `(1, 512, 256)` | float32 | grayscale X-ray, normalized to `[0, 1]` | |
| `seg` | `(512, 256)` | int64 | per-pixel class `0..17` | |
| `kpt_heatmap` | `(68, 128, 64)` | float32 | Gaussian blob per corner at H/4 × W/4 | zeros where vertebra missing |
| `kpt_valid` | `(68,)` | bool | mask of finite keypoints | warn cases (14–16 vert) have ≤16 valid groups |
| `cobb` | `()` | float32 | scalar Cobb angle (degrees) | NaN when missing |
| `cobb_valid` | `()` | bool | True if `cobb` is finite | only ~178 / 249 cases |

Heatmaps live at H/4 × W/4 (= 128 × 64) to keep the keypoint output tensor under 10 MB per sample. Argmax + ×4 scaling recovers pixel coordinates.

In [7]:
HEATMAP_DOWNSAMPLE = 4
HM_H = IMG_H // HEATMAP_DOWNSAMPLE  # 128
HM_W = IMG_W // HEATMAP_DOWNSAMPLE  # 64
HEATMAP_SIGMA = 2.0  # gaussian sigma in heatmap-resolution pixels

def render_heatmaps(
    keypoints_xy_full: np.ndarray,
    h: int = HM_H,
    w: int = HM_W,
    downsample: int = HEATMAP_DOWNSAMPLE,
    sigma: float = HEATMAP_SIGMA,
) -> tuple[np.ndarray, np.ndarray]:
    """Return (heatmaps (K,h,w) float32, valid (K,) bool)."""
    k = keypoints_xy_full.shape[0]
    hm = np.zeros((k, h, w), dtype=np.float32)
    valid = np.zeros((k,), dtype=bool)
    yy, xx = np.mgrid[0:h, 0:w].astype(np.float32)
    for i in range(k):
        x_full, y_full = keypoints_xy_full[i]
        if not (np.isfinite(x_full) and np.isfinite(y_full)):
            continue
        cx = x_full / downsample
        cy = y_full / downsample
        hm[i] = np.exp(-((xx - cx) ** 2 + (yy - cy) ** 2) / (2.0 * sigma ** 2))
        valid[i] = True
    return hm, valid


def keypoints_from_heatmaps(
    hm: torch.Tensor,
    threshold: float = 0.1,
    upsample: int = HEATMAP_DOWNSAMPLE,
) -> tuple[np.ndarray, np.ndarray]:
    """Argmax peak per channel, scale to full-resolution pixel coords.

    Args:
        hm: (K, h, w) heatmap tensor
    Returns:
        coords: (K, 2) float array of (x, y) in full-resolution pixels (NaN if peak < threshold)
        scores: (K,) peak intensities
    """
    k, h, w = hm.shape
    flat = hm.reshape(k, -1)
    scores, idx = flat.max(dim=1)
    ys = (idx // w).float() * upsample
    xs = (idx % w).float() * upsample
    coords = torch.stack([xs, ys], dim=1).cpu().numpy()
    s = scores.cpu().numpy()
    coords[s < threshold] = np.nan
    return coords, s

## 1.5 · Cobb-Preserving Augmentation (lifted from v3)


In [8]:
def augment_v5_cobb_aware(image, seg):
    """Same chain as augment_v4 but returns a `cobb_preserved` flag that is
    False when elastic_deform fires (which can change the spinal curve).
    Horizontal flip preserves |Cobb|, so it does NOT mask the Cobb target."""
    cobb_preserved = True
    if torch.rand(1).item() < 0.7:
        image, seg = affine_transform(
            image, seg,
            angle_range=(-15.0, 15.0),
            translate_frac=0.10,
            scale_range=(0.85, 1.15),
        )
    if torch.rand(1).item() < 0.5:
        image, seg = horizontal_flip(image, seg)
    if torch.rand(1).item() < 0.4:
        image, seg = elastic_deform(image, seg, alpha=40.0, sigma=6.0)
        cobb_preserved = False
    if torch.rand(1).item() < 0.5:
        image = intensity_jitter(image, gain_range=(0.7, 1.3), bias_range=(-0.15, 0.15))
    if torch.rand(1).item() < 0.3:
        image = gamma_correction(image)
    if torch.rand(1).item() < 0.3:
        image = gaussian_blur(image)
    if torch.rand(1).item() < 0.2:
        image = clahe_like(image)
    if torch.rand(1).item() < 0.3:
        image = gaussian_noise(image, std_range=(0.0, 0.08))
    if torch.rand(1).item() < 0.25:
        image = coarse_dropout(image, n_patches=12, patch_size=40)
    return image, seg, cobb_preserved


## 2 · Multi-Task Dataset

Wraps `preprocess_case` and adds heatmap rendering + Cobb extraction. Augmentation uses `augment_v5_cobb_aware` on `(image, seg)`, then **re-extracts keypoints from the augmented seg** so they always agree spatially. Cobb GT is invariant under flips and small affine, but **elastic deform** can change the spinal curve — the augmenter returns a `cobb_preserved` flag and we mask Cobb when elastic fires (cheaper than re-deriving post-deform GT).

In [9]:
class MultiTaskSpineDataset(Dataset):
    def __init__(
        self,
        df: pd.DataFrame,
        augment: bool = False,
        target_ids: tuple[int, ...] = TARGET_IDS_V2,
    ):
        self.df = df.reset_index(drop=True)
        self.augment = augment
        self.target_ids = target_ids

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, i: int) -> dict[str, torch.Tensor]:
        row = self.df.iloc[i]
        case = preprocess_case(row, target_ids=self.target_ids)
        image, seg = case["image"], case["seg"]

        cobb_preserved = True
        if self.augment:
            image, seg, cobb_preserved = augment_v5_cobb_aware(image, seg)

        # re-extract keypoints from (possibly augmented) seg so they spatially agree
        seg_np = seg.numpy().astype(np.uint8)
        # remapped label space already matches target_ids index → recover raw vertebra IDs
        # by indexing back: 0=bg, 1..17 → target_ids[0..16]
        # multiclass_mask_to_keypoints expects raw IDs, so we rebuild a raw-id mask:
        raw = np.zeros_like(seg_np)
        for k_idx, raw_id in enumerate(self.target_ids, start=1):
            raw[seg_np == k_idx] = raw_id
        kps_full = multiclass_mask_to_keypoints(raw, target_ids=self.target_ids)
        hm, valid = render_heatmaps(kps_full)

        cobb_val = row.get("cobb_angle_deg")
        has_gt = pd.notna(cobb_val) and cobb_preserved
        cobb_t = torch.tensor(float(cobb_val) if has_gt else 0.0, dtype=torch.float32)
        cobb_valid = torch.tensor(bool(has_gt))

        return {
            "image": image,
            "seg": seg,
            "kpt_heatmap": torch.from_numpy(hm),
            "kpt_valid": torch.from_numpy(valid),
            "cobb": cobb_t,
            "cobb_valid": cobb_valid,
        }


def multitask_collate(batch: list[dict]) -> dict[str, torch.Tensor]:
    return {k: torch.stack([b[k] for b in batch]) for k in batch[0]}


# sanity check one batch
_ds = MultiTaskSpineDataset(TRAINABLE.head(4), augment=False)
_b = multitask_collate([_ds[i] for i in range(4)])
for k, v in _b.items():
    print(f"  {k:14s}  {tuple(v.shape)}  {v.dtype}")


  image           (4, 1, 512, 256)  torch.float32
  seg             (4, 512, 256)  torch.int64
  kpt_heatmap     (4, 68, 128, 64)  torch.float32
  kpt_valid       (4, 68)  torch.bool
  cobb            (4,)  torch.float32
  cobb_valid      (4,)  torch.bool


## 3 · MultiTask Architecture

ResNet-34 encoder (ImageNet-pretrained, grayscale stem) shared across three heads:

- **Seg head**  — 1×1 conv on full-resolution decoder features → 18 logits per pixel (identical to `EncoderUNet`)
- **Kpt head**  — 1×1 conv on H/4 decoder features → 68 heatmaps (matches dataset target resolution, no extra upsample)
- **Cobb head** — global average pool on the bottleneck (B, 512, H/32, W/32) → 2-layer MLP → scalar (degrees)

Encoder/decoder topology mirrors `ai/models/architectures/encoder_unet.py`. Decoder features are tapped at two resolutions (H/4 for keypoints, H for seg).

In [10]:
def _conv_block(in_ch: int, out_ch: int) -> nn.Sequential:
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, 3, padding=1),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_ch, out_ch, 3, padding=1),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(inplace=True),
    )


class _Up(nn.Module):
    def __init__(self, in_ch: int, skip_ch: int, out_ch: int):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, in_ch // 2, 2, stride=2)
        self.conv = _conv_block(in_ch // 2 + skip_ch, out_ch)

    def forward(self, x: torch.Tensor, skip: torch.Tensor) -> torch.Tensor:
        x = self.up(x)
        dy = skip.shape[2] - x.shape[2]
        dx = skip.shape[3] - x.shape[3]
        if dy != 0 or dx != 0:
            x = F.pad(x, [dx // 2, dx - dx // 2, dy // 2, dy - dy // 2])
        return self.conv(torch.cat([x, skip], dim=1))


class MultiTaskEncoderUNet(nn.Module):
    """Shared ResNet-34 encoder with seg + keypoint + Cobb heads."""

    def __init__(
        self,
        in_ch: int = 1,
        num_seg_classes: int = NUM_SEG_CLASSES,
        num_keypoints: int = TOTAL_KEYPOINTS,
        pretrained: bool = True,
        dropout: float = 0.2,
        cobb_norm: float = 90.0,
    ):
        super().__init__()
        self.cobb_norm = cobb_norm

        weights = models.ResNet34_Weights.IMAGENET1K_V1 if pretrained else None
        backbone = models.resnet34(weights=weights)

        original_conv1 = backbone.conv1
        self.conv1 = nn.Conv2d(in_ch, 64, kernel_size=7, stride=2, padding=3, bias=False)
        if pretrained and in_ch == 1:
            self.conv1.weight.data = original_conv1.weight.data.mean(dim=1, keepdim=True)
        elif pretrained and in_ch == 3:
            self.conv1.weight.data = original_conv1.weight.data

        self.bn1 = backbone.bn1
        self.relu = backbone.relu
        self.maxpool = backbone.maxpool
        self.layer1 = backbone.layer1
        self.layer2 = backbone.layer2
        self.layer3 = backbone.layer3
        self.layer4 = backbone.layer4

        self.up1 = _Up(512, 256, 256)
        self.up2 = _Up(256, 128, 128)
        self.up3 = _Up(128, 64, 64)   # produces (B, 64, H/4, W/4) — keypoint tap
        self.up4 = _Up(64, 64, 64)    # produces (B, 64, H/2, W/2)
        self.final_up = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)
        self.drop = nn.Dropout2d(dropout)

        # heads
        self.seg_head = nn.Conv2d(64, num_seg_classes, 1)
        self.kpt_head = nn.Sequential(
            nn.Conv2d(64, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, num_keypoints, 1),
        )
        self.cobb_head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(512, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(128, 1),
        )

        self._encoder_modules = [
            self.conv1, self.bn1, self.layer1, self.layer2, self.layer3, self.layer4,
        ]
        self._decoder_modules = [
            self.up1, self.up2, self.up3, self.up4, self.final_up, self.drop,
            self.seg_head, self.kpt_head, self.cobb_head,
        ]

    def forward(self, x: torch.Tensor) -> dict[str, torch.Tensor]:
        x0 = self.relu(self.bn1(self.conv1(x)))
        x_pool = self.maxpool(x0)
        s1 = self.layer1(x_pool)
        s2 = self.layer2(s1)
        s3 = self.layer3(s2)
        bridge = self.layer4(s3)  # (B, 512, H/32, W/32) — Cobb tap

        y = self.up1(bridge, s3)
        y = self.up2(y, s2)
        y_q = self.up3(y, s1)              # (B, 64, H/4, W/4) — kpt tap
        y = self.up4(y_q, x0)
        y_full = self.final_up(y)          # (B, 64, H, W)
        y_full = self.drop(y_full)

        seg_logits = self.seg_head(y_full)
        kpt_heatmaps = self.kpt_head(y_q)  # (B, 68, H/4, W/4)
        cobb_norm_pred = self.cobb_head(bridge).squeeze(-1)  # (B,)
        cobb_pred = cobb_norm_pred * self.cobb_norm  # rescale to degrees

        return {
            "seg": seg_logits,
            "kpt": kpt_heatmaps,
            "cobb": cobb_pred,
        }

    def encoder_params(self):
        for m in self._encoder_modules:
            yield from m.parameters()

    def decoder_params(self):
        for m in self._decoder_modules:
            yield from m.parameters()


def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())


model = MultiTaskEncoderUNet(pretrained=True, dropout=0.2).to(DEVICE)
n_enc = sum(p.numel() for p in model.encoder_params())
n_dec = sum(p.numel() for p in model.decoder_params())
print(f"MultiTaskEncoderUNet total params: {count_params(model):,}")
print(f"  encoder: {n_enc:,}  decoder+heads: {n_dec:,}")

MultiTaskEncoderUNet total params: 24,501,303
  encoder: 21,278,400  decoder+heads: 3,222,903


Dropped Escape call with ulEscapeCode : 0x03007703


## 5 · Multi-Task Loss

Three raw losses, combined via `MixedLoss` = **clamped Kendall** uncertainty weighting on `(seg, cobb)` PLUS **fixed-weight** kpt:

- **L_seg** — `seg_loss_fn` (CE + soft Dice, bg-weighted), identical to v3
- **L_kpt** — masked MSE on heatmaps; loss averaged only over valid (vertebra-present) channels
- **L_cobb** — masked smooth-L1 on `(pred - gt) / cobb_norm`; skipped when no batch sample has Cobb GT

$$L = \tfrac{1}{2}e^{-s_\text{seg}}\,L_\text{seg} + \tfrac{1}{2}s_\text{seg} + \tfrac{1}{2}e^{-s_\text{cobb}}\,L_\text{cobb} + \tfrac{1}{2}s_\text{cobb} + \lambda_\text{kpt}\,L_\text{kpt}$$

Log-variances `s_seg, s_cobb` clamped to `[-1.5, 4.0]`. `λ_kpt = 1.0` fixed — kpt is held in the supervision graph as a regularizer, not a Cobb estimator (v2 showed Kendall amplification of the heatmap loss flips Cobb correlation).

In [11]:
COBB_NORM = 90.0
LAMBDA_KPT = 1.0
LOG_VAR_MIN = -1.5
LOG_VAR_MAX = 4.0


def kpt_loss_fn(pred: torch.Tensor, target: torch.Tensor, valid: torch.Tensor) -> torch.Tensor:
    """Masked MSE. valid: (B, K) bool; pred/target: (B, K, h, w)."""
    mask = valid.float().unsqueeze(-1).unsqueeze(-1)
    diff = (pred - target) ** 2 * mask
    denom = mask.sum() * pred.shape[-1] * pred.shape[-2] + 1e-6
    return diff.sum() / denom


def cobb_loss_fn(
    pred_deg: torch.Tensor,
    target_deg: torch.Tensor,
    valid: torch.Tensor,
    norm: float = COBB_NORM,
) -> torch.Tensor:
    if not bool(valid.any()):
        return pred_deg.sum() * 0.0
    p = pred_deg[valid] / norm
    t = target_deg[valid] / norm
    return F.smooth_l1_loss(p, t)


class ClampedKendallLoss(nn.Module):
    """Kendall '17 task uncertainty weighting with clamped log-variances.

    Verified offline + in v3: clamping `log_var ∈ [-1.5, 4.0]` prevents the
    σ-blowup that broke v2's naïve Kendall when small-magnitude losses were
    in play.
    """

    def __init__(
        self,
        task_names: tuple[str, ...] = ("seg", "cobb"),
        log_var_min: float = LOG_VAR_MIN,
        log_var_max: float = LOG_VAR_MAX,
    ):
        super().__init__()
        self.task_names = task_names
        self.log_var = nn.Parameter(torch.zeros(len(task_names)))
        self.log_var_min = log_var_min
        self.log_var_max = log_var_max

    def forward(self, raws: dict[str, torch.Tensor]) -> tuple[torch.Tensor, dict[str, float]]:
        clamped = self.log_var.clamp(self.log_var_min, self.log_var_max)
        total = 0.0
        parts: dict[str, float] = {}
        for i, name in enumerate(self.task_names):
            lv = clamped[i]
            li = raws[name]
            term = 0.5 * torch.exp(-lv) * li + 0.5 * lv
            total = total + term
            parts[f"raw_{name}"] = float(li.detach().cpu())
            parts[f"log_var_{name}"] = float(self.log_var[i].detach().cpu())
            parts[f"weight_{name}"] = float((0.5 * torch.exp(-lv)).detach().cpu())
        return total, parts


class MixedLoss(nn.Module):
    """v4 loss: clamped Kendall over (seg, cobb) PLUS fixed-weight kpt MSE.

    The kpt term is held at a fixed weight on purpose — v2 proved that letting
    Kendall amplify the heatmap loss flips Cobb correlation. Here we keep kpt
    in the supervision graph as a regularizer, not a Cobb estimator.
    """

    def __init__(self, kpt_weight: float = LAMBDA_KPT) -> None:
        super().__init__()
        self.kendall = ClampedKendallLoss(("seg", "cobb"))
        self.kpt_weight = kpt_weight

    def forward(
        self,
        raws: dict[str, torch.Tensor],
    ) -> tuple[torch.Tensor, dict[str, float]]:
        kendall_total, parts = self.kendall({"seg": raws["seg"], "cobb": raws["cobb"]})
        kpt_term = self.kpt_weight * raws["kpt"]
        total = kendall_total + kpt_term
        parts["raw_kpt"] = float(raws["kpt"].detach().cpu())
        parts["weight_kpt"] = self.kpt_weight
        parts["loss"] = float(total.detach().cpu())
        return total, parts


def raw_losses_for(out: dict, batch: dict, dev: torch.device) -> dict[str, torch.Tensor]:
    return {
        "seg": seg_loss_fn(out["seg"], batch["seg"].to(dev), NUM_SEG_CLASSES),
        "kpt": kpt_loss_fn(
            out["kpt"],
            batch["kpt_heatmap"].to(dev),
            batch["kpt_valid"].to(dev),
        ),
        "cobb": cobb_loss_fn(
            out["cobb"],
            batch["cobb"].to(dev),
            batch["cobb_valid"].to(dev),
        ),
    }


## 7 · Per-Fold Training Loop

Loop over the 5 folds. For each fold: fresh `MultiTaskEncoderUNet`, train with v4 recipe (encoder freeze warmup 10 ep, cosine LR, dual-checkpoint best-dice + best-cobb, patience 25). Per-fold cache.

**Resumability**: if a fold's checkpoint dir already exists with `metrics.json`, that fold is skipped on rerun. Lets a failed mid-run resume.

**Optional EMA**: maintains `torch.optim.swa_utils.AveragedModel(decay=0.999)` alongside the regular model. Per-fold EMA checkpoint saved as `model_ema.pt`. Set `USE_EMA = False` to disable (saves a small amount of memory + a couple seconds per epoch).


In [12]:
import hashlib
import json
from datetime import datetime

CHECKPOINT_ROOT = REPO_ROOT / "ai" / "models" / "checkpoints" / "multitask_v4_5fold"
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)

NUM_EPOCHS = 150
BATCH_SIZE = 4
LR_ENCODER = 1e-4
LR_DECODER = 1e-3
LR_LOGVAR = 1e-3
WEIGHT_DECAY = 2e-4
WARMUP_EPOCHS = 10
PATIENCE = 25
DROPOUT = 0.2
USE_EMA = True              # Tier-1 easy win per [[Easy_Wins_Backlog]] T1.2
EMA_DECAY = 0.999


def run_config() -> dict:
    return {
        "experiment": "multitask_v4_5fold_cv",
        "resolves": "Q-01",
        "dataset": "v2_corrected",
        "arch": "MultiTaskEncoderUNetV4",
        "encoder": "resnet34",
        "num_seg_classes": NUM_SEG_CLASSES,
        "num_keypoints": TOTAL_KEYPOINTS,
        "heatmap_h": HM_H,
        "heatmap_w": HM_W,
        "heatmap_sigma": HEATMAP_SIGMA,
        "loss": "mixed_clamped_kendall_seg_cobb_plus_fixed_kpt",
        "lambda_kpt": LAMBDA_KPT,
        "log_var_min": LOG_VAR_MIN,
        "log_var_max": LOG_VAR_MAX,
        "cobb_norm": COBB_NORM,
        "augment": "v5_cobb_aware",
        "preprocessing": "bare_div255",
        "num_epochs": NUM_EPOCHS,
        "batch_size": BATCH_SIZE,
        "lr_encoder": LR_ENCODER,
        "lr_decoder": LR_DECODER,
        "lr_logvar": LR_LOGVAR,
        "weight_decay": WEIGHT_DECAY,
        "warmup_epochs": WARMUP_EPOCHS,
        "patience": PATIENCE,
        "dropout": DROPOUT,
        "seed": SEED,
        "trainable_count": int(len(TRAINABLE)),
        "n_folds": N_FOLDS,
        "use_ema": USE_EMA,
        "ema_decay": EMA_DECAY,
    }


def cfg_hash(cfg: dict) -> str:
    return hashlib.sha256(json.dumps(cfg, sort_keys=True, default=str).encode()).hexdigest()[:8]


def find_or_create_run_dir(cfg: dict) -> Path:
    h = cfg_hash(cfg)
    matches = sorted(CHECKPOINT_ROOT.glob(f"*_{h}"))
    if matches:
        return matches[-1]  # most recent matching
    stamp = datetime.now().strftime("%Y%m%dT%H%M")
    p = CHECKPOINT_ROOT / f"{stamp}_{h}"
    p.mkdir(parents=True, exist_ok=True)
    return p


CFG = run_config()
RUN_DIR = find_or_create_run_dir(CFG)
(RUN_DIR / "config.json").write_text(json.dumps(CFG, indent=2, default=str))
print(f"run dir: {RUN_DIR.relative_to(REPO_ROOT)}  hash={cfg_hash(CFG)}")


run dir: ai/models/checkpoints/multitask_v4_5fold/20260503T1715_249cfc78  hash=249cfc78


In [13]:
def train_step(model, loss_module, batch, optimizer, ema_model=None):
    out = model(batch["image"].to(DEVICE))
    raws = raw_losses_for(out, batch, DEVICE)
    total, parts = loss_module(raws)
    optimizer.zero_grad()
    total.backward()
    optimizer.step()
    if ema_model is not None:
        ema_model.update_parameters(model)
    return parts


@torch.no_grad()
def validate(model, loss_module, loader):
    model.eval()
    accum = {"loss": 0.0, "raw_seg": 0.0, "raw_kpt": 0.0, "raw_cobb": 0.0,
             "weight_seg": 0.0, "weight_cobb": 0.0}
    inter = torch.zeros(NUM_SEG_CLASSES - 1, device=DEVICE)
    card = torch.zeros(NUM_SEG_CLASSES - 1, device=DEVICE)
    cobb_abs_err = []
    n_batches = 0
    for batch in loader:
        out = model(batch["image"].to(DEVICE))
        raws = raw_losses_for(out, batch, DEVICE)
        total, parts = loss_module(raws)
        accum["loss"] += float(total.detach().cpu())
        for k in ("raw_seg", "raw_kpt", "raw_cobb", "weight_seg", "weight_cobb"):
            accum[k] += parts[k]
        pred_mask = out["seg"].argmax(dim=1)
        seg_t = batch["seg"].to(DEVICE)
        for c in range(1, NUM_SEG_CLASSES):
            p, g = (pred_mask == c), (seg_t == c)
            inter[c - 1] += (p & g).sum()
            card[c - 1] += p.sum() + g.sum()
        valid = batch["cobb_valid"]
        if valid.any():
            cobb_pred = out["cobb"].cpu()[valid]
            cobb_true = batch["cobb"][valid]
            cobb_abs_err.extend((cobb_pred - cobb_true).abs().tolist())
        n_batches += 1
    dice_per_cls = (2.0 * inter) / card.clamp(min=1e-6)
    valid_cls = card > 0
    return {
        **{k: v / max(1, n_batches) for k, v in accum.items()},
        "dice": float(dice_per_cls[valid_cls].mean()) if valid_cls.any() else float("nan"),
        "cobb_mae_direct": float(np.mean(cobb_abs_err)) if cobb_abs_err else float("nan"),
    }


In [14]:
def train_one_fold(fold_idx, train_idx, val_idx):
    """Train v4 recipe on one fold. Returns (best_dice, best_cobb_mae, history_df, fold_dir).

    Skips if `metrics.json` already exists in the fold dir (resumability).
    """
    fold_dir = RUN_DIR / f"fold_{fold_idx}"
    fold_dir.mkdir(parents=True, exist_ok=True)
    metrics_path = fold_dir / "metrics.json"
    if metrics_path.exists():
        print(f"\n=== fold {fold_idx} — already trained, loading cached metrics ===")
        cached = json.loads(metrics_path.read_text())
        history_df = pd.read_csv(fold_dir / "history.csv")
        return cached["best_val_dice"], cached.get("best_val_cobb_mae", float("inf")), history_df, fold_dir

    print(f"\n=== fold {fold_idx} — training (n_train={len(train_idx)}, n_val={len(val_idx)}) ===")
    train_df = TRAINABLE.iloc[train_idx].reset_index(drop=True)
    val_df   = TRAINABLE.iloc[val_idx].reset_index(drop=True)

    train_loader = DataLoader(MultiTaskSpineDataset(train_df, augment=True),
                              batch_size=BATCH_SIZE, shuffle=True, collate_fn=multitask_collate)
    val_loader = DataLoader(MultiTaskSpineDataset(val_df, augment=False),
                            batch_size=BATCH_SIZE, shuffle=False, collate_fn=multitask_collate)

    model = MultiTaskEncoderUNet(pretrained=True, dropout=DROPOUT).to(DEVICE)
    loss_module = MixedLoss().to(DEVICE)
    ema_model = None
    if USE_EMA:
        ema_model = torch.optim.swa_utils.AveragedModel(
            model,
            multi_avg_fn=torch.optim.swa_utils.get_ema_multi_avg_fn(EMA_DECAY),
        )

    for p in model.encoder_params():
        p.requires_grad = False
    optimizer = torch.optim.Adam([
        {"params": model.decoder_params(), "lr": LR_DECODER, "weight_decay": WEIGHT_DECAY},
        {"params": loss_module.parameters(), "lr": LR_LOGVAR, "weight_decay": 0.0},
    ])
    scheduler = None

    history = []
    best_dice = -1.0
    best_cobb_mae = float("inf")
    best_dice_state = None
    best_cobb_state = None
    best_dice_epoch = -1
    best_cobb_epoch = -1
    no_improve = 0

    t_start = time.time()
    for epoch in range(1, NUM_EPOCHS + 1):
        t0 = time.time()

        if epoch == WARMUP_EPOCHS + 1:
            for p in model.encoder_params():
                p.requires_grad = True
            optimizer = torch.optim.Adam([
                {"params": model.encoder_params(), "lr": LR_ENCODER, "weight_decay": WEIGHT_DECAY},
                {"params": model.decoder_params(), "lr": LR_DECODER, "weight_decay": WEIGHT_DECAY},
                {"params": loss_module.parameters(), "lr": LR_LOGVAR, "weight_decay": 0.0},
            ])
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                optimizer, T_max=NUM_EPOCHS - WARMUP_EPOCHS
            )
            print(f"  >>> encoder unfrozen at epoch {epoch}")

        model.train()
        train_acc = {"loss": 0.0, "raw_seg": 0.0, "raw_kpt": 0.0, "raw_cobb": 0.0}
        n_steps = 0
        for batch in train_loader:
            parts = train_step(model, loss_module, batch, optimizer, ema_model=ema_model)
            train_acc["loss"] += parts["loss"]
            for k in ("raw_seg", "raw_kpt", "raw_cobb"):
                train_acc[k] += parts[k]
            n_steps += 1
        train_acc = {k: v / max(1, n_steps) for k, v in train_acc.items()}

        val_stats = validate(model, loss_module, val_loader)
        if scheduler is not None:
            scheduler.step()

        n_groups = len(optimizer.param_groups)
        row = {
            "epoch": epoch,
            "train_loss": train_acc["loss"],
            "train_seg": train_acc["raw_seg"],
            "train_kpt": train_acc["raw_kpt"],
            "train_cobb": train_acc["raw_cobb"],
            "val_loss": val_stats["loss"],
            "val_seg": val_stats["raw_seg"],
            "val_kpt": val_stats["raw_kpt"],
            "val_cobb": val_stats["raw_cobb"],
            "w_seg": val_stats["weight_seg"],
            "w_cobb": val_stats["weight_cobb"],
            "val_dice": val_stats["dice"],
            "val_cobb_mae": val_stats["cobb_mae_direct"],
            "sec": time.time() - t0,
        }
        history.append(row)

        improved_dice = val_stats["dice"] > best_dice
        improved_cobb = (
            not math.isnan(val_stats["cobb_mae_direct"])
            and val_stats["cobb_mae_direct"] < best_cobb_mae
        )
        if improved_dice:
            best_dice = val_stats["dice"]
            best_dice_state = {
                "model": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "loss_module": {k: v.detach().cpu().clone() for k, v in loss_module.state_dict().items()},
            }
            best_dice_epoch = epoch
            no_improve = 0
        else:
            no_improve += 1
        if improved_cobb:
            best_cobb_mae = val_stats["cobb_mae_direct"]
            best_cobb_state = {
                "model": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "loss_module": {k: v.detach().cpu().clone() for k, v in loss_module.state_dict().items()},
            }
            best_cobb_epoch = epoch

        flags = (" *D" if improved_dice else "") + (" *C" if improved_cobb else "")
        print(f"  ep {epoch:3d}/{NUM_EPOCHS}  tr={row['train_loss']:6.3f}  val={row['val_loss']:6.3f}  "
              f"dice={row['val_dice']:.3f}  cobb={row['val_cobb_mae']:5.1f}°  ({row['sec']:.1f}s){flags}")

        if epoch > WARMUP_EPOCHS and no_improve >= PATIENCE:
            print(f"  >>> early stop at epoch {epoch}")
            break

    history_df = pd.DataFrame(history)
    if best_dice_state is not None:
        torch.save(best_dice_state, fold_dir / "model.pt")
    if best_cobb_state is not None:
        torch.save(best_cobb_state, fold_dir / "model_best_cobb.pt")
    if ema_model is not None:
        torch.save({"model": {k: v.detach().cpu().clone() for k, v in ema_model.module.state_dict().items()}},
                   fold_dir / "model_ema.pt")
    history_df.to_csv(fold_dir / "history.csv", index=False)
    metrics = {
        "fold": fold_idx,
        "best_val_dice": best_dice,
        "best_val_dice_epoch": best_dice_epoch,
        "best_val_cobb_mae": best_cobb_mae if best_cobb_mae != float("inf") else None,
        "best_val_cobb_mae_epoch": best_cobb_epoch,
        "stopped_epoch": int(history_df["epoch"].iloc[-1]),
        "final_log_var": loss_module.kendall.log_var.detach().cpu().tolist(),
        "elapsed_sec": time.time() - t_start,
    }
    metrics_path.write_text(json.dumps(metrics, indent=2))
    print(f"  fold {fold_idx} done in {metrics['elapsed_sec']:.1f}s  "
          f"best dice={best_dice:.3f}@ep{best_dice_epoch}  "
          f"best cobb={best_cobb_mae:.2f}°@ep{best_cobb_epoch}")
    return best_dice, best_cobb_mae, history_df, fold_dir


In [15]:
all_fold_results = []
for fold_idx, (train_idx, val_idx) in enumerate(FOLD_SPLITS):
    best_dice, best_cobb, history_df, fold_dir = train_one_fold(fold_idx, train_idx, val_idx)
    all_fold_results.append({
        "fold": fold_idx,
        "n_val": len(val_idx),
        "best_dice": best_dice,
        "best_cobb_mae": best_cobb,
        "fold_dir": str(fold_dir.relative_to(REPO_ROOT)),
    })

RESULTS_DF = pd.DataFrame(all_fold_results)
RESULTS_DF.to_csv(RUN_DIR / "per_fold_results.csv", index=False)
print("\n=== ALL FOLDS COMPLETE ===")
print(RESULTS_DF.to_string(index=False))



=== fold 0 — already trained, loading cached metrics ===

=== fold 1 — already trained, loading cached metrics ===

=== fold 2 — already trained, loading cached metrics ===

=== fold 3 — already trained, loading cached metrics ===

=== fold 4 — already trained, loading cached metrics ===

=== ALL FOLDS COMPLETE ===
 fold  n_val  best_dice  best_cobb_mae                                                               fold_dir
    0     50   0.669728       7.339891 ai/models/checkpoints/multitask_v4_5fold/20260503T1715_249cfc78/fold_0
    1     50   0.723556       8.715518 ai/models/checkpoints/multitask_v4_5fold/20260503T1715_249cfc78/fold_1
    2     50   0.643993       8.635926 ai/models/checkpoints/multitask_v4_5fold/20260503T1715_249cfc78/fold_2
    3     50   0.636953       8.149580 ai/models/checkpoints/multitask_v4_5fold/20260503T1715_249cfc78/fold_3
    4     49   0.627446       7.968620 ai/models/checkpoints/multitask_v4_5fold/20260503T1715_249cfc78/fold_4


## 8 · Aggregate Metrics — Mean ± SD Across Folds

**The headline thesis number.** Reports `dice = mean ± SD` and `cobb_direct MAE = mean ± SD` across the 5 folds. This is what goes in the chapter.


In [16]:
summary = {
    "n_folds": int(len(RESULTS_DF)),
    "dice_mean": float(RESULTS_DF["best_dice"].mean()),
    "dice_std": float(RESULTS_DF["best_dice"].std()),
    "dice_min": float(RESULTS_DF["best_dice"].min()),
    "dice_max": float(RESULTS_DF["best_dice"].max()),
    "cobb_mae_mean": float(RESULTS_DF["best_cobb_mae"].mean()),
    "cobb_mae_std": float(RESULTS_DF["best_cobb_mae"].std()),
    "cobb_mae_min": float(RESULTS_DF["best_cobb_mae"].min()),
    "cobb_mae_max": float(RESULTS_DF["best_cobb_mae"].max()),
}
(RUN_DIR / "summary.json").write_text(json.dumps(summary, indent=2))

print("=== Multi-Task v4 Kendall — 5-Fold CV (Q-01 thesis-number gate) ===\n")
print(f"  n_folds:       {summary['n_folds']}")
print(f"  dice (per-fold best, mean ± SD):       {summary['dice_mean']:.3f} ± {summary['dice_std']:.3f}")
print(f"    range: [{summary['dice_min']:.3f}, {summary['dice_max']:.3f}]")
print(f"  cobb_direct MAE (per-fold best, mean ± SD):  {summary['cobb_mae_mean']:.2f}° ± {summary['cobb_mae_std']:.2f}°")
print(f"    range: [{summary['cobb_mae_min']:.2f}°, {summary['cobb_mae_max']:.2f}°]")

print("\nReference (single 80/20):")
print("  v4 Kendall:  dice 0.680  cobb_direct MAE 5.79°")
print("\nGate (per the notebook header):")
print("  cobb mean ≈ 5.79° ± 1° → single-split was representative; lock as architecture ceiling")
print("  cobb mean ≈ 5.79° but SD > 1.5° → high variance; ceiling honest at the higher SD bound")
print("  cobb mean significantly worse (>7°) → single-split overestimated; revise expectations")


=== Multi-Task v4 Kendall — 5-Fold CV (Q-01 thesis-number gate) ===

  n_folds:       5
  dice (per-fold best, mean ± SD):       0.660 ± 0.039
    range: [0.627, 0.724]
  cobb_direct MAE (per-fold best, mean ± SD):  8.16° ± 0.56°
    range: [7.34°, 8.72°]

Reference (single 80/20):
  v4 Kendall:  dice 0.680  cobb_direct MAE 5.79°

Gate (per the notebook header):
  cobb mean ≈ 5.79° ± 1° → single-split was representative; lock as architecture ceiling
  cobb mean ≈ 5.79° but SD > 1.5° → high variance; ceiling honest at the higher SD bound
  cobb mean significantly worse (>7°) → single-split overestimated; revise expectations


## 9 · K-Fold Ensemble Inference (optional, free win per [[Multi_stage_pipelines]] §C)

For each val sample (across all folds), run inference through ALL 5 fold-models and average the predictions. This is the deployment number — what we'd report if we had a separate held-out test set.

Note: this is mildly improper because each fold's model was trained on data that includes the OTHER folds' val samples. So the ensemble is inflated for those samples. The honest interpretation is "upper bound on what an ensemble could achieve if folds were independent." For a strict deployment metric, you'd need to also hold out a separate test set during folding (which we didn't here because n=152 is already small).


In [ ]:
@torch.no_grad()
def evaluate_kfold_ensemble(fold_dirs, val_df, ckpt_name="model.pt"):
    """Average predictions from all K fold-models on val_df. Returns dice and cobb_direct MAE."""
    K = len(fold_dirs)
    fold_models = []
    for fd in fold_dirs:
        m = MultiTaskEncoderUNet(pretrained=False, dropout=DROPOUT).to(DEVICE)
        state = torch.load(fd / ckpt_name, map_location='cpu')
        m.load_state_dict(state["model"])
        m.eval()
        fold_models.append(m)

    inter = torch.zeros(NUM_SEG_CLASSES - 1, device=DEVICE)
    card = torch.zeros(NUM_SEG_CLASSES - 1, device=DEVICE)
    cobb_abs_err = []
    for _, row in val_df.iterrows():
        case = preprocess_case(row)
        img = case["image"].unsqueeze(0).to(DEVICE)
        seg_logits_sum = None
        cobb_pred_sum = 0.0
        for m in fold_models:
            out = m(img)
            if seg_logits_sum is None:
                seg_logits_sum = out["seg"].clone()
            else:
                seg_logits_sum += out["seg"]
            cobb_pred_sum += float(out["cobb"].squeeze(0).cpu())
        seg_pred = (seg_logits_sum / K).argmax(dim=1).squeeze(0)
        cobb_pred = cobb_pred_sum / K

        seg_t = case["seg"].to(DEVICE)
        for c in range(1, NUM_SEG_CLASSES):
            p, g = (seg_pred == c), (seg_t == c)
            inter[c - 1] += (p & g).sum()
            card[c - 1] += p.sum() + g.sum()
        gt = row.get("cobb_angle_deg")
        if pd.notna(gt):
            cobb_abs_err.append(abs(cobb_pred - float(gt)))
    dice_per_cls = (2.0 * inter) / card.clamp(min=1e-6)
    valid = card > 0
    return {
        "dice": float(dice_per_cls[valid].mean()) if valid.any() else float("nan"),
        "cobb_mae": float(np.mean(cobb_abs_err)) if cobb_abs_err else float("nan"),
    }


# pool ALL val samples across folds — each sample is in exactly one fold's val
all_val_idx = np.concatenate([va for _, va in FOLD_SPLITS])
ALL_VAL_DF = TRAINABLE.iloc[all_val_idx].reset_index(drop=True)
FOLD_DIRS = [RUN_DIR / f"fold_{k}" for k in range(N_FOLDS)]

print("=== K-Fold Ensemble Inference (best-dice ckpts) ===")
ens_best_dice = evaluate_kfold_ensemble(FOLD_DIRS, ALL_VAL_DF, ckpt_name="model.pt")
print(f"  dice = {ens_best_dice['dice']:.3f}")
print(f"  cobb_direct MAE = {ens_best_dice['cobb_mae']:.2f}°")

print("\n=== K-Fold Ensemble Inference (best-cobb ckpts) ===")
ens_best_cobb = evaluate_kfold_ensemble(FOLD_DIRS, ALL_VAL_DF, ckpt_name="model_best_cobb.pt")
print(f"  dice = {ens_best_cobb['dice']:.3f}")
print(f"  cobb_direct MAE = {ens_best_cobb['cobb_mae']:.2f}°")

if USE_EMA:
    print("\n=== K-Fold Ensemble Inference (EMA ckpts, best-dice timepoint) ===")
    ens_ema = evaluate_kfold_ensemble(FOLD_DIRS, ALL_VAL_DF, ckpt_name="model_ema.pt")
    print(f"  dice = {ens_ema['dice']:.3f}")
    print(f"  cobb_direct MAE = {ens_ema['cobb_mae']:.2f}°")

ENSEMBLE_RESULTS = {
    "best_dice_ckpt": ens_best_dice,
    "best_cobb_ckpt": ens_best_cobb,
    "ema_ckpt": ens_ema if USE_EMA else None,
}
(RUN_DIR / "ensemble_results.json").write_text(json.dumps(ENSEMBLE_RESULTS, indent=2))


=== K-Fold Ensemble Inference (best-dice ckpts) ===


/tmp/ipykernel_3381/232811701.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(fd / ckpt_name, map_location='cpu')


  dice = 0.897
  cobb_direct MAE = 3.15°

=== K-Fold Ensemble Inference (best-cobb ckpts) ===
  dice = 0.872
  cobb_direct MAE = 4.25°

=== K-Fold Ensemble Inference (EMA ckpts, best-dice timepoint) ===
  dice = 0.879
  cobb_direct MAE = 3.90°


273

: 

## 10 · Test-Time Augmentation (TTA, hflip-only — free 2× ensemble)

Per [[Easy_Wins_Backlog]] T1.1. Horizontal flip preserves `|Cobb|`, so it's a free 2× ensemble at inference. Average seg logits before argmax; average cobb_direct prediction.

Restricted to hflip — rotation augmentations would change the Cobb angle and require compensation.


In [ ]:
@torch.no_grad()
def evaluate_tta_per_fold(fold_dirs, val_dfs, ckpt_name="model.pt"):
    """For each fold, evaluate its OWN val set with hflip TTA. Returns per-fold dice + cobb_mae."""
    rows = []
    for k, (fd, val_df) in enumerate(zip(fold_dirs, val_dfs)):
        m = MultiTaskEncoderUNet(pretrained=False, dropout=DROPOUT).to(DEVICE)
        state = torch.load(fd / ckpt_name, map_location='cpu')
        m.load_state_dict(state["model"])
        m.eval()
        inter = torch.zeros(NUM_SEG_CLASSES - 1, device=DEVICE)
        card = torch.zeros(NUM_SEG_CLASSES - 1, device=DEVICE)
        cobb_abs_err = []
        for _, row in val_df.iterrows():
            case = preprocess_case(row)
            img = case["image"].unsqueeze(0).to(DEVICE)
            img_flipped = torch.flip(img, dims=[-1])
            out = m(img)
            out_flip = m(img_flipped)
            # un-flip the seg logits before averaging
            seg_avg = (out["seg"] + torch.flip(out_flip["seg"], dims=[-1])) / 2.0
            cobb_avg = (out["cobb"].squeeze(0).cpu().item() + out_flip["cobb"].squeeze(0).cpu().item()) / 2.0
            seg_pred = seg_avg.argmax(dim=1).squeeze(0)
            seg_t = case["seg"].to(DEVICE)
            for c in range(1, NUM_SEG_CLASSES):
                p, g = (seg_pred == c), (seg_t == c)
                inter[c - 1] += (p & g).sum()
                card[c - 1] += p.sum() + g.sum()
            gt = row.get("cobb_angle_deg")
            if pd.notna(gt):
                cobb_abs_err.append(abs(cobb_avg - float(gt)))
        dice_per_cls = (2.0 * inter) / card.clamp(min=1e-6)
        valid = card > 0
        rows.append({
            "fold": k,
            "dice_tta": float(dice_per_cls[valid].mean()) if valid.any() else float("nan"),
            "cobb_mae_tta": float(np.mean(cobb_abs_err)) if cobb_abs_err else float("nan"),
        })
    return pd.DataFrame(rows)


fold_val_dfs = [TRAINABLE.iloc[va].reset_index(drop=True) for _, va in FOLD_SPLITS]
print("=== TTA per-fold (best-dice ckpts) ===")
TTA_DF = evaluate_tta_per_fold(FOLD_DIRS, fold_val_dfs, ckpt_name="model.pt")
TTA_DF.to_csv(RUN_DIR / "per_fold_tta_results.csv", index=False)
print(TTA_DF.to_string(index=False))
print()
print(f"  dice_tta:    mean={TTA_DF['dice_tta'].mean():.3f} ± {TTA_DF['dice_tta'].std():.3f}")
print(f"  cobb_mae_tta: mean={TTA_DF['cobb_mae_tta'].mean():.2f}° ± {TTA_DF['cobb_mae_tta'].std():.2f}°")
print()
print(f"  vs baseline (no TTA):")
print(f"    Δdice (TTA - no-TTA): {TTA_DF['dice_tta'].mean() - summary['dice_mean']:+.3f}")
print(f"    Δcobb_mae (TTA - no-TTA): {TTA_DF['cobb_mae_tta'].mean() - summary['cobb_mae_mean']:+.2f}°")


## 11 · Conclusions

TODO after the run: fill in the headline numbers and update the wiki.

**Headline thesis numbers** (from §8):
- `v4 Kendall 5-fold CV: dice = ___ ± ___, cobb_direct MAE = ___° ± ___°`

**Bonus numbers** (from §9 and §10):
- `K-fold ensemble (best-dice): dice = ___, cobb_direct MAE = ___°`
- `K-fold ensemble (best-cobb): dice = ___, cobb_direct MAE = ___°`
- `K-fold ensemble (EMA):       dice = ___, cobb_direct MAE = ___°`
- `Per-fold TTA: dice mean = ___, cobb_direct MAE mean = ___°`

**Decision tree** (from notebook header):
- cobb mean ≈ 5.79° ± 1° → single-split was representative; **lock 5.79° as architecture ceiling**
- cobb mean ≈ 5.79° but SD > 1.5° → high variance; **ceiling honest at the higher SD bound**
- cobb mean significantly worse (> 7°) → single-split overestimated; **revise published expectations**

**File the result** as `[[2026-05-XX_v4_5fold_cv]]` with:
- Per-fold table (5 rows)
- Aggregate (mean ± SD)
- Comparison to single-split v4 (5.79° / 0.680)
- TTA delta (if positive, becomes default inference)
- K-fold ensemble number (deployment-style, with caveat about train-on-other-folds-val inflation)
- EMA delta (if positive, default training step for future runs)
- Update [[experiments/_index|Experiments index]] §Leaderboard, [[hot]] §Last Updated, [[Open_Questions]] Q-01 → resolved
